In [ ]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 81 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 2s (411 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
import os

log_file = open("/content/ollama.log", "w")

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Ollama PID:", ollama_process.pid)

Ollama PID: 3673


In [ ]:
import requests
import time

for i in range(20):
    try:
        r = requests.get("http://127.0.0.1:11434", timeout=2)
        print(r.text)
        break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    print("Ollama chưa khởi động được")

Ollama is running


In [ ]:
!pip install ollama

In [ ]:
!ollama pull qwen2.5:3b

In [ ]:
import pandas as pd

df = pd.read_json('/content/playlist_dataset.json')
df.head()

,videoId,title,start,seconds,text_chunk,summary
0,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:00:00,0,Xin chào các bạn Nếu các bạn là những người đa...,
1,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:03:00,180,vào phần Chính các bạn nhá linear regression l...,
2,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:06:02,362,ới với trình độ là CIO Giả sử chúng ta vẫn cố ...,
3,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:09:04,544,xem một con vật nó là con hổ hay là con sư tử ...,
4,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:12:06,726,ra sự kiện A Biết rằng sự kiện B đã xảy ra rồi...,


In [ ]:
def summary(text):
    response = ollama.chat(
        model="qwen2.5:3b",
        messages=[
            {
                "role": "system",
                "content": """Bạn là trợ lý tóm tắt văn bản.
Tóm tắt đoạn văn bản sau bằng tiếng Việt trong tối đa 60 từ.
CHỈ TRẢ VỀ DUY NHẤT ĐOẠN TÓM TẮT. Không giải thích, không suy nghĩ ra ngoài, không viết tiếng Anh."""
            },
            {
                "role": "user",
                "content": f"Văn bản cần tóm tắt:\n{text}"
            }
        ],
        options={
            "temperature": 0,
            "num_predict": 250,        # Tăng token để chứa đủ bản tóm tắt
            "think": False             # Tắt reasoning trong options (nếu phiên bản Ollama hỗ trợ)
        }
    )

    # Lấy nội dung trả về
    content = response['message']['content'].strip()

    # Xử lý triệt để nếu model vẫn cố tình trả về thẻ <think>...</think>
    if "<think>" in content and "</think>" in content:
        content = content.split("</think>")[-1].strip()

    return content

In [ ]:


df['summary'] = df['text_chunk'].apply(summary)

df.head()

,videoId,title,start,seconds,text_chunk,summary
0,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:00:00,0,Xin chào các bạn Nếu các bạn là những người đa...,Xin chào các bạn. Video này tổng hợp và tóm tắ...
1,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:03:00,180,vào phần Chính các bạn nhá linear regression l...,Linear regression là một thuật toán cơ bản tro...
2,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:06:02,362,ới với trình độ là CIO Giả sử chúng ta vẫn cố ...,Đây là tóm tắt đoạn văn bản trong 60 từ:\n\nĐể...
3,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:09:04,544,xem một con vật nó là con hổ hay là con sư tử ...,Xây dựng và huấn luyện một mô hình phân loại h...
4,ULE78ME1ckQ,All Machine Learning Algorithms in 23 Minutes,00:12:06,726,ra sự kiện A Biết rằng sự kiện B đã xảy ra rồi...,Xác suất một email là email spam biết rằng nó ...


In [ ]:
len(df['summary'])
print(df.shape)

(510, 6)


In [ ]:

df.to_json("youtube_dataset.json", orient="records", indent=4, force_ascii=False)

print("Đã xuất thành công file youtube_dataset.json!")

Đã xuất thành công file youtube_dataset.json!
